# Assessment 1 - Airbnb Price Prediction Pipeline

## Student Details
- Name: [Your Name]
- Student Number: sXXXXXXX
- Course: COSC2673 / COSC2793

## Objective
Build and evaluate a supervised machine learning pipeline to predict Airbnb nightly listing price (`price`) from listing, host, and location-related features.

## Reproducibility
This notebook uses a fixed random seed (`RANDOM_STATE = 42`) and a deterministic pipeline structure to improve reproducibility of results.

## How To Run
1. Place `train_data.csv` and `test_data.csv` in the same folder as this notebook.
2. Run all cells from top to bottom.
3. Replace `student_number` in the prediction cell before exporting your CSV.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')


## Task Definition and Evaluation Framework

- **Task type**: Supervised regression.
- **Target variable**: `price` (nightly listing price).
- **Why regression**: `price` is continuous and should be estimated from available features.

### Evaluation Design
- Hold-out split: 80% training, 20% validation.
- Stability estimate: 5-fold cross-validation on training split.
- Metrics:
  - MAE: average absolute error in price units (easy business interpretation).
  - RMSE: penalizes large errors more strongly than MAE.
  - R2: proportion of variance explained by the model.

The final test set remains unseen until final model selection to avoid optimistic bias.

In [ ]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
display(train_df.head())
display(train_df.dtypes.to_frame('dtype'))

display(train_df.describe(include='all').T)
print('Missing values (train):')
display(train_df.isna().sum()[train_df.isna().sum() > 0])


In [ ]:
# Missingness profile
missing_counts = train_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]

if len(missing_counts) > 0:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=missing_counts.index, y=missing_counts.values, color='steelblue')
    plt.xticks(rotation=45, ha='right')
    plt.title('Missing values per feature (train)')
    plt.ylabel('Count of missing values')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values in training data.')


## Initial Data Interpretation

The dataset contains **8,586 training rows** and **8,585 test rows**. The training data has 16 columns (15 predictors + 1 target `price`), while the test data has 15 predictor columns.

### Feature types
- Categorical: `host_is_superhost`, `city`, `country`, `room_type`, `instant_bookable`
- Numerical: `latitude`, `longitude`, `accommodates`, `bathrooms`, `bedrooms`, `beds`, `minimum_nights`, `number_of_reviews`, `review_scores_rating`, `calculated_host_listings_count`

### Missingness
No missing values were detected in the provided training set at runtime. Imputation remains in the pipeline as a robust safeguard for reproducibility and for potential future data with incomplete records.

### Validity note
Even with complete data, learned relationships remain correlational. Pricing can also be influenced by unobserved factors (seasonality, event periods, listing quality details), which limits causal interpretation.

In [ ]:
X = train_df.drop(columns=['price'])
y = train_df['price']

categorical_features = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print('Categorical:', categorical_features)
print('Numeric:', numeric_features)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, kde=True, ax=axes[0])
axes[0].set_title('Price distribution')
sns.boxplot(x=y, ax=axes[1])
axes[1].set_title('Price boxplot')
plt.tight_layout()
plt.show()

corr = train_df[numeric_features + ['price']].corr(numeric_only=True)
plt.figure(figsize=(10, 6))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Numeric correlation heatmap')
plt.tight_layout()
plt.show()


## EDA Interpretation

The `price` distribution is right-skewed with visible high-price outliers, indicating that a minority of listings are substantially more expensive than typical listings. This motivates using both MAE and RMSE: MAE for average error magnitude, RMSE for sensitivity to large misses.

From the correlation heatmap, location and capacity-related variables (for example `accommodates`, `bedrooms`, and geographic coordinates) show meaningful linear relationships with `price`, though strengths vary and are not sufficient alone for perfect prediction.

All available features are retained initially to avoid premature feature dropping. Regularized models (Ridge/Lasso) then provide controlled complexity and help reduce the impact of weak or redundant predictors.

Correlation results are used for interpretation and model design only; they do not establish causation.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ]
)

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

def evaluate(name, model):
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)

    val_pred = pipe.predict(X_val)
    val_m = metrics(y_val, val_pred)

    cv = cross_validate(
        pipe, X_train, y_train, cv=kfold,
        scoring=('neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'),
        n_jobs=-1
    )

    return {
        'Model': name,
        'Val_MAE': val_m['MAE'],
        'Val_RMSE': val_m['RMSE'],
        'Val_R2': val_m['R2'],
        'CV_MAE': -cv['test_neg_mean_absolute_error'].mean(),
        'CV_RMSE': -cv['test_neg_root_mean_squared_error'].mean(),
        'CV_R2': cv['test_r2'].mean()
    }, pipe


## Why This Pipeline Design

- `ColumnTransformer` ensures each feature type gets appropriate preprocessing.
- Median imputation is robust for skewed numerical features.
- Most-frequent categorical imputation is simple and consistent for low missingness.
- One-hot encoding makes categorical values usable by linear models.
- Standardization is important for fair regularization in Ridge/Lasso.
- Wrapping preprocessing + model in a single pipeline reduces leakage risk.

In [ ]:
# Model 1: Linear Regression
linear_row, linear_pipe = evaluate('LinearRegression', LinearRegression())
display(pd.DataFrame([linear_row]))


In [ ]:
# Model 2: Ridge (tune alpha)
ridge_alphas = np.logspace(-3, 3, 13)
ridge_rows = []

for a in ridge_alphas:
    row, _ = evaluate(f'Ridge(alpha={a:.4g})', Ridge(alpha=float(a), random_state=RANDOM_STATE))
    row['alpha'] = a
    ridge_rows.append(row)

ridge_df = pd.DataFrame(ridge_rows).sort_values('Val_RMSE').reset_index(drop=True)
best_ridge_alpha = float(ridge_df.loc[0, 'alpha'])
display(ridge_df.head())

plt.figure(figsize=(6, 4))
plt.plot(ridge_df['alpha'], ridge_df['Val_RMSE'], marker='o')
plt.xscale('log')
plt.title('Ridge alpha vs Validation RMSE')
plt.xlabel('alpha')
plt.ylabel('Val RMSE')
plt.tight_layout()
plt.show()


In [ ]:
# Model 3: Lasso (tune alpha)
lasso_alphas = np.logspace(-4, 1, 12)
lasso_rows = []

for a in lasso_alphas:
    row, _ = evaluate(f'Lasso(alpha={a:.4g})', Lasso(alpha=float(a), random_state=RANDOM_STATE, max_iter=20000))
    row['alpha'] = a
    lasso_rows.append(row)

lasso_df = pd.DataFrame(lasso_rows).sort_values('Val_RMSE').reset_index(drop=True)
best_lasso_alpha = float(lasso_df.loc[0, 'alpha'])
display(lasso_df.head())

plt.figure(figsize=(6, 4))
plt.plot(lasso_df['alpha'], lasso_df['Val_RMSE'], marker='o', color='orange')
plt.xscale('log')
plt.title('Lasso alpha vs Validation RMSE')
plt.xlabel('alpha')
plt.ylabel('Val RMSE')
plt.tight_layout()
plt.show()


## Hyperparameter Analysis (Ridge/Lasso Alpha)

Hyperparameter tuning shows the expected bias-variance pattern:
- Very small `alpha` behaves similarly to ordinary least squares (weak regularization).
- Very large `alpha` overshrinks coefficients and can underfit.
- Intermediate values balance generalization and stability.

Using validation RMSE for ranking and CV metrics for stability checks, the best-performing candidate in this run was **Lasso(alpha = 0.152)**.

Top validation result:
- Val_MAE = 47.0066
- Val_RMSE = 103.5609
- Val_R2 = 0.3969

Corresponding CV result:
- CV_MAE = 47.2447
- CV_RMSE = 90.5979
- CV_R2 = 0.4327

In [ ]:
comparison = pd.DataFrame([
    linear_row,
    ridge_df.loc[0].to_dict(),
    lasso_df.loc[0].to_dict()
]).sort_values('Val_RMSE').reset_index(drop=True)

display(comparison[['Model', 'Val_MAE', 'Val_RMSE', 'Val_R2', 'CV_MAE', 'CV_RMSE', 'CV_R2']])
best_model_name = comparison.loc[0, 'Model']
print('Selected model:', best_model_name)

if best_model_name.startswith('LinearRegression'):
    selected_estimator = LinearRegression()
elif best_model_name.startswith('Ridge'):
    selected_estimator = Ridge(alpha=best_ridge_alpha, random_state=RANDOM_STATE)
else:
    selected_estimator = Lasso(alpha=best_lasso_alpha, random_state=RANDOM_STATE, max_iter=20000)

# Error analysis on validation set
analysis_pipe = Pipeline([('preprocess', preprocessor), ('model', selected_estimator)])
analysis_pipe.fit(X_train, y_train)
val_pred = analysis_pipe.predict(X_val)
residuals = y_val - val_pred

plt.figure(figsize=(7, 4))
sns.scatterplot(x=val_pred, y=residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted price')
plt.ylabel('Residual (actual - predicted)')
plt.title('Residual plot on validation set')
plt.tight_layout()
plt.show()

# Feature importance via coefficients (for linear family)
importance_pipe = Pipeline([('preprocess', preprocessor), ('model', selected_estimator)])
importance_pipe.fit(X, y)
model = importance_pipe.named_steps['model']
feature_names = importance_pipe.named_steps['preprocess'].get_feature_names_out()
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': model.coef_})
coef_df = coef_df.sort_values('coefficient')
print('Most negative coefficients:')
display(coef_df.head(10))
print('Most positive coefficients:')
display(coef_df.tail(10))

# Train final model on full training data and predict test set
final_pipeline = Pipeline([('preprocess', preprocessor), ('model', selected_estimator)])
final_pipeline.fit(X, y)
test_pred = np.clip(final_pipeline.predict(test_df), a_min=0, a_max=None)

student_number = 'sXXXXXXX'  # replace with your student number
out_file = f'{student_number}_predictions.csv'
pd.DataFrame({'price': test_pred}).to_csv(out_file, index=False)
print('Saved prediction file:', out_file)
pred_df = pd.DataFrame({'price': test_pred})
display(pred_df.head())
print('Prediction shape:', pred_df.shape)
assert pred_df.shape[0] == test_df.shape[0]
assert list(pred_df.columns) == ['price']

print('CSV format checks passed.')


## Final Analysis, Real-World Suitability, and Ethics

### Model adequacy for practice
The selected model is suitable as a **decision-support tool** for initial price recommendations, not as a fully autonomous pricing decision system. Error magnitudes indicate useful directional guidance, but not perfect precision for all listings.

### Error behavior and limitations
- Residual analysis should be checked for systematic over/underestimation across price ranges.
- Linear-family models may miss nonlinear interactions in real markets.
- The dataset is a processed subset and may not represent all Melbourne market conditions or time periods.
- Omitted variables (seasonality, event demand, listing quality details) can reduce predictive performance.

### Feature importance (COSC2793 focus)
Coefficient-based interpretation from the selected linear model provides directional insights into which transformed features increase or decrease predicted price. Interpretation must be cautious because coefficients are affected by scaling and one-hot encoding reference levels.

### Ethical and professional responsibilities
- Historical data may contain socioeconomic or geographic bias; model outputs can propagate this bias.
- Users should be informed about uncertainty and expected error.
- Final accountability for pricing decisions remains with the human decision-maker.
- Data must be handled responsibly and only for the assignment's permitted scope.

### Conclusion
Lasso(alpha = 0.152) was selected due to best validation RMSE among evaluated linear-family models with competitive cross-validation performance. The model is practically useful for baseline recommendations, with clear limitations and ethical constraints that should be acknowledged in deployment contexts.